# Compare results of SWITi and inner tiling

## Gradient Test

#### Load Images

In [ ]:
import numpy as np
import os
import ndv
import tifffile as tiff

In [ ]:
DATASET = "HT_LIF24_500ms"
IMG_IDX = 0

In [ ]:
root_dir = f"/project/careamics/switi/results/{DATASET}/predictions_MMSE64"

In [ ]:
inner_tiling_data = np.load(os.path.join(root_dir, "inner_tiling", "predictions.npz"), allow_pickle=True)
switi_data = np.load(os.path.join(root_dir, "sw_inner_tiling", "predictions.npz"), allow_pickle=True)

print(switi_data.files, inner_tiling_data.files)

FNAME = switi_data.files[IMG_IDX]
switi_img = switi_data[FNAME].squeeze()
inner_tiling_img = inner_tiling_data[FNAME].squeeze()
# gt_img = tiff.imread(f"/project/careamics/switi/data/{DATASET}/targets/test/{FNAME}.tif")
gt_img = tiff.imread(f"/project/careamics/switi/data/{DATASET}/targets/test/{FNAME.replace('input', 'target')}.tif")

switi_img.shape, inner_tiling_img.shape, gt_img.shape

In [ ]:
v1 = ndv.imshow(switi_img)

In [ ]:
v2 = ndv.imshow(inner_tiling_img)

In [ ]:
v3 = ndv.imshow(gt_img)

Save crop for figure about Permutation Test

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
crop = inner_tiling_img[0, 3, 832:896]

fig = plt.imshow(crop, cmap="gray")
plt.savefig("./PaviaATN_inner_tiling_test_img01_crop_ch0_y_384-448_x_832-896.pdf", dpi=600)

#### Check gradient maps

In [ ]:
from analysis_pipeline.gradient_test.gradient_analysis import compute_gradients
from analysis_pipeline.gradient_test.plotting import plot_gradient_comparison

In [ ]:
CHANNEL = 0

In [ ]:
inner_tiling_grads = compute_gradients(inner_tiling_img[CHANNEL])
switi_grads = compute_gradients(switi_img[CHANNEL])
gt_grads = compute_gradients(gt_img[CHANNEL])
[grad.shape for grad in inner_tiling_grads], [grad.shape for grad in switi_grads], [grad.shape for grad in gt_grads]

In [ ]:
fig = plot_gradient_comparison(
    {
        "Inner Tiling": inner_tiling_grads,
        "SWiTi": switi_grads,
        "Ground Truth": gt_grads
    },
    y_ROI=(831, 896), #(150, 300), #(1300, 1600) # (1440,2416 1540)
    x_ROI=(383, 448), #(1450, 1600), #(1280, 1580) # (1310, 1410)
    save_path="./PaviaATN_inner_tiling_test_img01_crop_ch0_y_384-448_x_832-896_gradients.pdf",
    dpi=600
)

#### Run the gradient tests

In [ ]:
from analysis_pipeline.gradient_test.analysis import run_gradient_analysis

In [ ]:
inner_tiling_report = run_gradient_analysis(
    inner_tiling_img,
    tile_size=[64, 64],
    overlap=[32, 32],
    image_id=FNAME,
    channels=None,
    statistic="js",
    strip_width=2,
)

In [ ]:
switi_report = run_gradient_analysis(
    switi_img,
    tile_size=[64, 64],
    overlap=[32, 32],
    image_id=FNAME,
    channels=None,
    statistic="js",
    strip_width=2,
)

In [ ]:
gt_report = run_gradient_analysis(
    gt_img,
    tile_size=[64, 64],
    overlap=[32, 32],
    image_id=FNAME,
    channels=None,
    statistic="js",
    strip_width=2,
)

#### Plot results of the gradient tests

In [ ]:
from analysis_pipeline.gradient_test.plotting import (
    plot_pvalue_distribution,
    plot_significance_overlay,
    plot_significance_overlay_grid
)

In [ ]:
for i in range(switi_img.shape[0]):
    fig = plot_significance_overlay(
        switi_report,
        switi_img[i],
        channel=i,
        tile_size=[64, 64],
        overlap=[32, 32],
        title=f"SWITi - Ch. {i}",
        overlay_alpha=0.5,
        y_ROI=(660, 820),# (150, 300),# (1300, 1600) # (1440, 1540)
        x_ROI=(1250, 1410),# (1310, 1410) #(1280, 1580) # (1310, 1410)
        save_path=f"./images/HT_LIF24_500ms/gradient_test_HT_LIF24_500ms_SWITi_ch{i}_y660-820_x1250-1410.pdf",
        dpi=600
    )

In [ ]:
for i in range(inner_tiling_img.shape[0]):
    fig = plot_significance_overlay(
        inner_tiling_report,
        inner_tiling_img[i],
        channel=i,
        tile_size=[64, 64],
        overlap=[32, 32],
        title=f"Inner Tiling - Ch. {i}",
        overlay_alpha=0.5,
        y_ROI=(660, 820),# (150, 300),# (1300, 1600) # (1440, 1540)
        x_ROI=(1250, 1410),# (1310, 1410) #(1280, 1580) # (1310, 1410)
        save_path=f"./images/HT_LIF24_500ms/gradient_test_HT_LIF24_500ms_inner_tiling_ch{i}_y660-820_x1250-1410.pdf",
        dpi=600
    )

In [ ]:
for i in range(gt_img.shape[0]):
    fig = plot_significance_overlay(
        gt_report,
        gt_img[i],
        channel=i,
        tile_size=[64, 64],
        overlap=[32, 32],
        title=f"GT - Ch. {i}",
        overlay_alpha=0.5,
        y_ROI=(660, 820),# (150, 300),# (1300, 1600) # (1440, 1540)
        x_ROI=(1250, 1410),# (1310, 1410) #(1280, 1580) # (1310, 1410)
        save_path=f"./images/HT_LIF24_500ms/gradient_test_HT_LIF24_500ms_GT_ch{i}_y660-820_x1250-1410.pdf",
        dpi=600
    )

#### Check results of gradient test in flat regions

SWITi shows significant divergence in flat regions. That's probably because also small discrepancies sticks out in flat regions, as there is no natural variability in the image and so also a tiny gradient can be significant.

In [ ]:
inner_tiling_report_js = run_gradient_analysis(
    inner_tiling_img[np.newaxis, :, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    channel=0,
    statistic="js",
    save_dir=None
)

In [ ]:
fig = plot_significance_overlay(
    inner_tiling_report_js.images[0],
    inner_tiling_img[0, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    title="Inner Tiling",
)

In [ ]:
switi_report_js = run_gradient_analysis(
    switi_img[np.newaxis, :, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    channel=0,
    statistic="js",
    save_dir=None
)

In [ ]:
fig = plot_significance_overlay(
    switi_report_js.images[0],
    switi_img[0, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    title="SWITi",
)

In [ ]:
gt_report_js = run_gradient_analysis(
    gt_img[np.newaxis, :, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    channel=0,
    statistic="js",
    save_dir=None
)

In [ ]:
fig = plot_significance_overlay(
    gt_report_js.images[0],
    gt_img[0, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    title="SWITi",
)

## FRC metric

#### Load images

In [ ]:
import numpy as np
import os
import ndv
import tifffile as tiff

In [ ]:
DATASET = "PaviaATN"
IMG_IDX = 0

In [ ]:
root_dir = f"/project/careamics/switi/results/{DATASET}/predictions"

In [ ]:
inner_tiling_data = np.load(os.path.join(root_dir, "inner_tiling", "predictions.npz"), allow_pickle=True)
switi_data = np.load(os.path.join(root_dir, "sw_inner_tiling", "predictions.npz"), allow_pickle=True)

print(switi_data.files, inner_tiling_data.files)

FNAME = switi_data.files[IMG_IDX]
switi_img = switi_data[FNAME].squeeze()
inner_tiling_img = inner_tiling_data[FNAME].squeeze()
gt_img = tiff.imread(f"/project/careamics/switi/data/{DATASET}/targets/test/{FNAME}.tif")

switi_img.shape, inner_tiling_img.shape, gt_img.shape

In [ ]:
v1 = ndv.imshow(switi_img)

In [ ]:
v2 = ndv.imshow(inner_tiling_img)

In [ ]:
v3 = ndv.imshow(gt_img)

#### Run FRC metric on images

In [ ]:
from analysis_pipeline.frc.analysis import run_frc_analysis

In [ ]:
frc_report_switi = run_frc_analysis(
    switi_img[np.newaxis, ...],
    gt_img[np.newaxis, ...],
    save_dir=None,
    method_name="SWITi",
    channels=[0]
)

In [ ]:
frc_report_inner_tiling = run_frc_analysis(
    inner_tiling_img[np.newaxis, ...],
    gt_img[np.newaxis, ...],
    save_dir=None,
    method_name="inner_tiling",
    channels=[0]
)

In [ ]:
frc_report_switi

### Plot distribution for Figure about perm test

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np

In [ ]:
p_seam_obs = np.load("./seam_distrib_obs.npy")
p_control_obs = np.load("./control_distrib_obs.npy")
p_seam_perm = np.load("./seam_distrib_perm.npy")[0]
p_control_perm = np.load("./control_distrib_perm.npy")[0]
p_control_obs.sum(), p_control_perm.sum(), p_seam_obs.sum(), p_seam_perm.sum()

### Inspect the Gradient Permutation Test Results

In [ ]:
from pathlib import Path
import pandas as pd
from analysis_pipeline.gradient_test.aggregation import MethodReport

In [ ]:
DATADIR = Path("/project/careamics/switi/results/CARE3D_zebrafish/predictions_MMSE64/gradient_test/strip_width=1") 

In [ ]:
report_switi = MethodReport.load(DATADIR / "SWITi_gradient_report.json")
report_inner_tiling = MethodReport.load(DATADIR / "inner_tiling_gradient_report.json")

In [ ]:
records_inner_tiling = [row for row in report_inner_tiling.to_records()]
df_inner_tiling = pd.DataFrame.from_records(records_inner_tiling)
df_inner_tiling.head()

In [ ]:
records_switi = [row for row in report_switi.to_records()]
df_switi = pd.DataFrame.from_records(records_switi)
df_switi.head()

In [ ]:
csv_path = DATADIR / "summary_inner_tiling.csv"
df_inner_tiling.to_csv(csv_path, index=False)

In [ ]:
csv_path = DATADIR / "summary_switi.csv"
df_switi.to_csv(csv_path, index=False)